In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split, Subset
import torch.optim as optim
from torchvision import transforms
from sklearn.metrics import classification_report
from PIL import Image
from tqdm import tqdm

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.ids = df["id_code"].values
        self.labels = df["diagnosis"].values
        self.image_dir = image_dir
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.ids[idx] + ".png")
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [3]:
class FirstCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # convolution layers, stride is 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # max pooling + adaptive average pooling
        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        # fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [4]:
csv_path = "data/raw/aptos2019-blindness-detection/train.csv"
image_dir = "data/raw/aptos2019-blindness-detection/train_images"

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    # train mode
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    # progress bar with train dataloader
    loop = tqdm(loader, desc="  Train", leave=False)
    for images, labels in loop:
        # images + labels from dataloader
        images, labels = images.to(device), labels.to(device)

        # reset gradients
        optimizer.zero_grad()
        outputs = model(images)

        # calculate loss
        loss = criterion(outputs, labels)
        # backpropagate
        loss.backward()
        optimizer.step()

        # add losses, correct preds, total preds, to calculate accuracy
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
        loop.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct / total:.3f}")

    # return loss, accuracy
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    # evaluation mode
    model.eval()

    # keep track of loss, preds, and labels for accuracy + classification report
    total_loss = 0.0
    all_preds = []
    all_labels = []

    # turn off gradient opimization
    with torch.no_grad():
        # validation progress bar
        loop = tqdm(loader, desc="  Val  ", leave=False)
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            loop.set_postfix(loss=f"{loss.item():.4f}")

    total = len(all_labels)
    # how many times prediction == label
    correct = sum(p == l for p, l in zip(all_preds, all_labels))

    # print classification report
    print(classification_report(
        all_labels, all_preds,
        target_names=["No DR", "Mild", "Moderate", "Severe", "Proliferative"],
    ))
    
    return total_loss / total, correct / total

def train(
    model,
    csv_path=csv_path,
    image_dir=image_dir,
    num_epochs=10,
    batch_size=32,
    lr=1e-3,
    val_split=0.2,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # augmentation only applied at train time; fundus images have no canonical
    # orientation so flips/full rotation are valid, but hue is left alone since
    # color carries diagnostic signal (hemorrhages, exudates)
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=180),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # validation stays deterministic: resize only, no augmentation
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # train.csv to pandas dataframe
    df = pd.read_csv(csv_path)

    # split indices first so train/val can be backed by different transforms
    val_size = int(len(df) * val_split)
    train_size = len(df) - val_size
    train_indices, val_indices = random_split(range(len(df)), [train_size, val_size])

    train_set = Subset(BlindnessDataset(df, image_dir, transform=train_transform), train_indices.indices)
    val_set = Subset(BlindnessDataset(df, image_dir, transform=val_transform), val_indices.indices)

    # count classes and calculate weights for each class to handle class imbalance, on train split only
    train_labels = df['diagnosis'].values[train_set.indices]
    class_counts = pd.Series(train_labels).value_counts()
    class_weights = 1.0 / class_counts
    sample_weights = pd.Series(train_labels).map(class_weights).values

    # use weighted random sampler to sample from dataset with replacement based on class weights
    sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=train_size,
    replacement=True
    )

    # wrap datasets in dataloaders
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0, sampler=sampler)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0)

    model = model.to(device)
    # use adam optimizer and CE loss
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # progress bar for specific epoch
    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Epochs")
    for epoch in epoch_bar:
        # calculate training and validation loss, accuracy
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        # print loss + accuracy
        epoch_bar.write(
            f"Epoch {epoch:>2}/{num_epochs} | "
            f"train loss {train_loss:.4f}  acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f}  acc {val_acc:.3f}"
        )
        epoch_bar.set_postfix(
            val_loss=f"{val_loss:.4f}",
            val_acc=f"{val_acc:.3f}",
        )

    # save and return trained model
    torch.save(model.state_dict(), "model.pth")
    print("Model saved to model.pth")
    return model

In [17]:
train(FirstCNN(), num_epochs=10, batch_size=32, lr=1e-3, val_split=0.2)

Using device: cpu



Epochs:   0%|          | 0/10 [02:05<?, ?it/s]


KeyboardInterrupt: 